# Two-Dimensional Bond Percolation Model

In this tutorial, we analyze the 2D bond percolation problem using `jaxfss`. Unlike the Binder ratio in the Ising model, both the temperature-like parameter $p$ and the observable require non-zero scaling exponents ($c_1$ and $c_2$).


## 1. Theoretical Background

In 2D bond percolation on a square lattice, each edge is occupied with probability $p$. As $p$ approaches the critical threshold $p_{\mathrm{c}} = 1/2$, an infinite cluster emerges.

The mean cluster size (susceptibility) $\chi^{\mathrm{f}}(p, L)$ of finite clusters satisfies the finite-size scaling relation:

$$
\chi^{\mathrm{f}}(p, L) = L^{-c_2} F\left[ (p - p_{\mathrm{c}}) L^{c_1} \right]
$$

Known theoretical values for 2D percolation:
* $p_{\mathrm{c}} = 1/2 = 0.5$
* $c_1 = 1/\nu = 3/4 = 0.75$
* $c_2 = -\gamma/\nu = -7/4 = -1.75$


## 2. Load and Inspect Data

We load data from `docs/data/percolation.txt`.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import optax
from softclip import SoftClip

import jaxfss

dataset = jaxfss.CriticalData.from_file("data/percolation.txt")
print(f"Loaded {dataset.n_data} data points.")
print(f"System sizes: {jnp.unique(dataset.Ls)}")

Let's visualize the raw cluster susceptibility data.

In [ ]:
plt.figure(figsize=(7, 4.5))
unique_Ls = sorted(list(set(dataset.Ls.flatten().tolist())))
for L in unique_Ls:
    idx = dataset.Ls.flatten() == L
    plt.errorbar(
        dataset.Ts.flatten()[idx],
        dataset.As.flatten()[idx],
        yerr=dataset.As_err.flatten()[idx],
        fmt="o-",
        label=f"$L = {int(L)}$",
        capsize=2
    )

plt.xlabel(r"Occupation Probability $p$", fontsize=12)
plt.ylabel(r"Cluster Susceptibility $\chi^{\mathrm{f}}(p, L)$", fontsize=12)
plt.title("Raw Percolation Data", fontsize=14)
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Set Up Neural Scaling Analysis

For percolation, we estimate three critical parameters: $(c_1, c_2, p_{\mathrm{c}})$.
We constrain:
* $c_1 > 0$ via `SoftClip(low=0.0)`
* $c_2 < 0$ via `SoftClip(high=0.0)`
* Scaled $p_{\mathrm{c}} \in [-1, 1]$ via `SoftClip(low=-1.0, high=1.0)`

In [ ]:
mlp = jaxfss.RationalMLP(features=[20, 20, 1])
key = jax.random.PRNGKey(42)
mlp_params = mlp.init(key, jnp.ones((1, 1)))

bij_c1 = SoftClip(low=0.0)
bij_c2 = SoftClip(high=0.0)
bij_pc = SoftClip(low=-1.0, high=1.0)

init_params = {
    "mlp": mlp_params,
    "fss": jnp.zeros(3)  # [p_c1, p_c2, p_pc]
}

def get_fss_params(params):
    p1, p2, pc = params["fss"]
    c1 = bij_c1.forward(p1)
    c2 = bij_c2.forward(p2)
    scaled_pc = bij_pc.forward(pc)
    return c1, c2, scaled_pc

train_data = dataset.train_data
Ls = train_data["system_size"]
Ts = train_data["temperature"]
As = train_data["observable"]
vAs = train_data["observable_var"]

def loss_fn(params):
    c1, c2, scaled_pc = get_fss_params(params)
    X = (Ts - scaled_pc) * (Ls ** c1)
    Y = As * (Ls ** c2)
    V = vAs * (Ls ** (2 * c2))
    Y_pred = mlp.apply(params["mlp"], X)
    return jaxfss.NLLLoss(Y, Y_pred, V)

## 4. Optimization with NSA

We train the model with Adam using separate learning rates.

In [ ]:
optimizer = {
    "mlp": optax.adam(learning_rate=1e-3),
    "fss": optax.adam(learning_rate=1e-2)
}
steps = 8000

params, losses, critical_vals = jaxfss.fit(loss_fn, optimizer, init_params, steps)

c1_est, c2_est, scaled_pc_est = get_fss_params(params)
pc_est = float(dataset.bij_temperature.inverse(scaled_pc_est))
c1_est = float(c1_est)
c2_est = float(c2_est)

print("=== Estimation Results ===")
print(f"c_1 (exact = 0.75000)   : {c1_est:.5f}")
print(f"c_2 (exact = -1.75000)  : {c2_est:.5f}")
print(f"p_c (exact = 0.50000)   : {pc_est:.5f}")

## 5. Scaling Collapse and NN Fitting Curve

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4.5))

# Loss curve
ax1.plot(losses)
ax1.set_yscale("log")
ax1.set_xlabel("Step")
ax1.set_ylabel("NLL Loss")
ax1.set_title("Training Loss Curve")
ax1.grid(True, alpha=0.3)

# Scaling collapse
for L in unique_Ls:
    idx = dataset.Ls.flatten() == L
    t = Ts.flatten()[idx]
    s = Ls.flatten()[idx]
    x = (t - scaled_pc_est) * (s ** c1_est)
    y = As.flatten()[idx] * (s ** c2_est)
    yerr = jnp.sqrt(vAs.flatten()[idx]) * (s ** c2_est)
    ax2.errorbar(x, y, yerr=yerr, fmt="o", label=f"$L={int(L)}$", alpha=0.7, capsize=2)

# Plot NN prediction
x_grid = jnp.linspace(-1.0, 1.0, 200).reshape(-1, 1)
y_pred = mlp.apply(params["mlp"], x_grid)
ax2.plot(x_grid, y_pred, color="black", lw=2, linestyle="--", label="NN $F(X)$")

ax2.set_xlabel(r"Scaled probability $X = (p - p_{\mathrm{c}}) L^{c_1}$", fontsize=11)
ax2.set_ylabel(r"$Y = \chi^{\mathrm{f}}(p, L) L^{c_2}$", fontsize=11)
ax2.set_title(f"Data Collapse ($c_1={c1_est:.4f}, c_2={c2_est:.4f}, p_c={pc_est:.4f}$)", fontsize=12)
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()